# 04 — Robustesse au domain shift (Phase 5)

Trois variantes de prétraitement, comparées avec **exactement le même protocole d'entraînement** que la Phase 4 (ResNet18, phase A tête seule puis phase B layer3/layer4, mêmes hyperparamètres) :

- **A — augmentation de couleur** (`ColorJitter`, à l'entraînement seulement) ;
- **B — normalisation de coloration** (Reinhard, appliquée à l'entraînement ET à l'évaluation) ;
- **C — niveaux de gris** (supprime la couleur entièrement).

**Choix du modèle final sur `val_ood` (hôpital 1) uniquement** — comparé au modèle de la Phase 4 (`outputs/metrics/04_cnn.json`). L'hôpital 2 n'est pas touché ici non plus.

Voir `PLAN.md` (Phase 5) et `outputs/rapport/05_robustesse.md`.

In [ ]:
# Colab/Kaggle : clone le depot si besoin et place le repertoire de travail dans notebooks/
# (sans effet en local : le depot est deja la, donc rien n'est clone ni deplace)
import os, subprocess, sys

REPO_URL = "https://github.com/YoucefLgr/Breast_Cancer.git"
EN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if EN_COLAB or EN_KAGGLE:
    repo_dir = "/content/Breast_Cancer" if EN_COLAB else "/kaggle/working/Breast_Cancer"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    os.chdir(os.path.join(repo_dir, "notebooks"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)

print("Repertoire de travail :", os.getcwd())

In [ ]:
# Paramètres
QUICK = False   # True = sous-ensemble réduit, pour vérifier que tout tourne avant le run complet
VARIANTES = ["augmentation_couleur", "normalisation_coloration", "niveaux_gris"]

import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.utils import load_config, set_seed, get_device, detect_environment, evaluer_par_hopital, save_metrics, load_metrics, make_generator
from src.data import (charger_sous_echantillon, PatchDataset, construire_transforms,
                      calculer_cible_reinhard, NormalisationReinhard, ouvrir, HOPITAL_LABEL)
from src.model import construire_modele, geler_backbone, degeler_layers_superieures, compter_parametres
from src.train import entrainer, predire_probas

CFG = load_config()
SEED = CFG["seed"]
set_seed(SEED)
DEVICE = get_device()
print("Environnement :", detect_environment(), "| device :", DEVICE)

## 1. Données et cible de normalisation de coloration

La cible Reinhard est calculée une seule fois, sur 200 patchs de l'hôpital 0 (entraînement) — jamais sur `val_ood` ni sur l'hôpital 2.

In [ ]:
df = charger_sous_echantillon(CFG)
if QUICK:
    morceaux = [g.sample(min(len(g), 60), random_state=SEED)
                for _, g in df.groupby(["split", "center", "label"])]
    df = pd.concat(morceaux, ignore_index=True)
    print("Mode QUICK : sous-ensemble réduit à", len(df), "patchs")

df_train = df[df.split == "train"].reset_index(drop=True)
df_val = df[df.split == "val_ood"].reset_index(drop=True)
df_idval = df[df.split == "id_val"].reset_index(drop=True)

CIBLE_REINHARD = calculer_cible_reinhard(df[(df.split == "train") & (df.center == 0)], n=50 if QUICK else 200, seed=SEED)
print("Cible Reinhard (moyenne LAB, écart-type LAB) :", CIBLE_REINHARD[0].round(2), CIBLE_REINHARD[1].round(2))

In [ ]:
# Vérification visuelle : la normalisation Reinhard harmonise-t-elle bien les hôpitaux ?
norm = NormalisationReinhard(*CIBLE_REINHARD)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for h in range(5):
    s = df[df.center == h].sample(1, random_state=7)
    img = ouvrir(s.image.iloc[0])
    axes[0, h].imshow(img); axes[0, h].set_title(f"{HOPITAL_LABEL[h]}\noriginal", fontsize=8); axes[0, h].axis("off")
    axes[1, h].imshow(norm(img)); axes[1, h].set_title("Reinhard", fontsize=8); axes[1, h].axis("off")
plt.tight_layout()
plt.savefig("../outputs/figures/06_reinhard_avant_apres.png", dpi=130)
plt.show()

## 2. Entraînement des trois variantes

Même protocole que la Phase 4 pour chacune : phase A (tête seule) puis phase B (`layer3`/`layer4` + tête), mêmes hyperparamètres (`config.yaml`), sélection et arrêt anticipé sur la ROC-AUC de `val_ood`.

In [ ]:
def construire_dataloaders(variante):
    kw = {"cible_reinhard": CIBLE_REINHARD} if variante == "normalisation_coloration" else {}
    t_train, t_eval = construire_transforms(CFG, variante, **kw)
    gen = make_generator(SEED)
    bs = 32 if QUICK else CFG["modele"]["batch_size"]
    nw = 2 if DEVICE == "cuda" else 0
    dl_train = DataLoader(PatchDataset(df_train, t_train), batch_size=bs, shuffle=True,
                          generator=gen, num_workers=nw)
    dl_val = DataLoader(PatchDataset(df_val, t_eval), batch_size=bs, shuffle=False, num_workers=nw)
    dl_idval = DataLoader(PatchDataset(df_idval, t_eval), batch_size=bs, shuffle=False, num_workers=nw)
    return dl_train, dl_val, dl_idval

resultats_variantes = {}
for variante in VARIANTES:
    print(f"\n=========== Variante : {variante} ===========")
    set_seed(SEED)  # même initialisation et même ordre de batches pour chaque variante
    dl_train, dl_val, dl_idval = construire_dataloaders(variante)

    modele = construire_modele(CFG["modele"]["architecture"]).to(DEVICE)
    geler_backbone(modele)
    ep_tete = 1 if QUICK else CFG["modele"]["epochs_tete"]
    res_a = entrainer(modele, dl_train, dl_val, df_val.label.values, DEVICE, epochs=ep_tete,
                      lr=CFG["modele"]["lr_tete"], weight_decay=CFG["modele"]["weight_decay"],
                      patience=2, verbeux=True)

    degeler_layers_superieures(modele)
    ep_ft = 1 if QUICK else CFG["modele"]["epochs_finetune"]
    res_b = entrainer(modele, dl_train, dl_val, df_val.label.values, DEVICE, epochs=ep_ft,
                      lr=CFG["modele"]["lr_finetune"], weight_decay=CFG["modele"]["weight_decay"],
                      patience=2, checkpoint_path=f"../outputs/models/resnet18_{variante}.pt", verbeux=True)

    proba_val = predire_probas(modele, dl_val, DEVICE)
    proba_idval = predire_probas(modele, dl_idval, DEVICE)
    t_val = evaluer_par_hopital(df_val.label.values, proba_val, df_val.center.values, 0.5, HOPITAL_LABEL)
    t_idval = evaluer_par_hopital(df_idval.label.values, proba_idval, df_idval.center.values, 0.5, HOPITAL_LABEL)

    resultats_variantes[variante] = {
        "historique_phase_A": res_a["historique"], "historique_phase_B": res_b["historique"],
        "meilleur_auc_val_ood": res_b["meilleur_auc_val_ood"],
        "metriques_val_ood": t_val.to_dict(orient="records"),
        "metriques_id_val": t_idval.to_dict(orient="records"),
    }
    print(f"--> {variante} : meilleure AUC val_ood = {res_b['meilleur_auc_val_ood']:.4f}")

## 3. Comparaison à la Phase 4 (sans anti-shift) — AUC par hôpital, `train` et `val`

`outputs/metrics/04_cnn.json` (Phase 4) sert de repère « sans technique anti-shift ».

In [ ]:
# load_metrics (comme save_metrics) résout ses chemins depuis la racine du dépôt, pas le
# cwd du notebook : pas de "../" ici (contrairement à plt.savefig un peu plus bas).
baseline = load_metrics("outputs/metrics/04_cnn.json")

def ligne_comparaison(nom, enreg_idval, enreg_val):
    """Une ligne : AUC par hôpital d'entraînement, AUC ensemble id_val (repère biaisé),
    AUC val_ood (hôpital 1 — seule métrique qui compte pour le choix du modèle)."""
    par_hopital_id = {r["hopital"]: r["roc_auc"] for r in enreg_idval if r["hopital"] != "ensemble"}
    ensemble_id = next(r["roc_auc"] for r in enreg_idval if r["hopital"] == "ensemble")
    auc_val_ood = next(r["roc_auc"] for r in enreg_val if r["hopital"] == "ensemble")
    return {"variante": nom, **par_hopital_id, "entrainement (ensemble)": ensemble_id, "auc_val_ood": auc_val_ood}

lignes = [ligne_comparaison("baseline (Phase 4, aucune technique)",
                            baseline["metriques_seuil_0.5"]["id_val"], baseline["metriques_seuil_0.5"]["val_ood"])]
for variante, r in resultats_variantes.items():
    lignes.append(ligne_comparaison(variante, r["metriques_id_val"], r["metriques_val_ood"]))

comparaison = pd.DataFrame(lignes).set_index("variante")
cols_ordre = [c for c in [HOPITAL_LABEL[0], HOPITAL_LABEL[3], HOPITAL_LABEL[4],
                          "entrainement (ensemble)", "auc_val_ood"] if c in comparaison.columns]
display(comparaison[cols_ordre].round(4))

fig, ax = plt.subplots(figsize=(9, 4.5))
comparaison["auc_val_ood"].plot.barh(ax=ax, color="tab:blue")
ax.set_xlabel("ROC-AUC sur val_ood (hôpital 1)"); ax.set_xlim(0.8, 1.0)
ax.set_title("Effet des techniques anti-shift sur l'hôpital jamais vu")
plt.tight_layout(); plt.savefig("../outputs/figures/07_comparaison_robustesse.png", dpi=130); plt.show()

## 4. Écart entraînement ↔ hôpital 1 : la technique réduit-elle le domain shift ?

Pas seulement l'AUC absolue sur `val_ood` : l'**écart** entre la performance sur les hôpitaux d'entraînement (`id_val`) et sur l'hôpital 1 est la vraie mesure de robustesse au changement d'hôpital.

In [ ]:
# Écart entre l'AUC ensemble sur id_val (hôpitaux d'entraînement) et sur val_ood (hôpital 1) :
# un écart plus faible qu'à la baseline signale une technique qui réduit vraiment le domain shift,
# pas seulement une AUC val_ood plus haute en absolu.
comparaison["ecart_entrainement_moins_val_ood"] = comparaison["entrainement (ensemble)"] - comparaison["auc_val_ood"]
display(comparaison[["entrainement (ensemble)", "auc_val_ood", "ecart_entrainement_moins_val_ood"]]
        .round(4).sort_values("ecart_entrainement_moins_val_ood"))

## 5. Choix du modèle final (sur `val_ood` uniquement)

In [ ]:
meilleure_variante = comparaison["auc_val_ood"].idxmax()
print("Meilleure variante sur val_ood :", meilleure_variante, "-> AUC =", round(comparaison.loc[meilleure_variante, 'auc_val_ood'], 4))
print("Rappel, baseline Phase 4 : AUC =", round(comparaison.loc[comparaison.index.str.startswith('baseline'), 'auc_val_ood'].iloc[0], 4))

## 6. Sauvegarde des métriques

In [ ]:
metrics = {
    "quick_mode": QUICK,
    "variantes_testees": VARIANTES,
    "cible_reinhard": {"moyenne_lab": CIBLE_REINHARD[0].tolist(), "std_lab": CIBLE_REINHARD[1].tolist(),
                       "source": "200 patchs, hopital 0, split train"},
    "resultats_par_variante": resultats_variantes,
    "comparaison_auc": comparaison.reset_index().to_dict(orient="records"),
    "meilleure_variante_val_ood": meilleure_variante,
    "baseline_phase4_auc_val_ood": comparaison.loc[comparaison.index.str.startswith("baseline"), "auc_val_ood"].iloc[0],
}
print(save_metrics(metrics, "outputs/metrics/05_robustesse.json"))

## 7. Lecture des résultats

Voir `outputs/rapport/05_robustesse.md` : la technique retenue (ou l'absence de gain net) sera reprise comme modèle candidat pour le test unique sur l'hôpital 2 (Phase 6).